In [2]:
import formulallm.formula as f

import io
import contextlib

from llama_index.core.tools import FunctionTool
from llama_index.core.agent import ReActAgent, FunctionCallingAgentWorker
from llama_index.llms.ollama import Ollama
from llama_index.core import PromptTemplate

In [3]:
llama3 = Ollama(model="llama3", base_url='http://localhost:11434', temperature=0.0, request_timeout=600)

In [4]:
def load(filePath: str) -> str:
    """Load the Formula DSL code from the file path"""
    code = f.load(filePath)
    return code

load_tool = FunctionTool.from_defaults(fn=load)

In [5]:
def solve(partial_model:str, domain: str):
    """Try to solve the partial model based on the domain constraints"""
    """You can get the name of the partial model and the domain from the loaded file"""
    stdout_buffer = io.StringIO()
    stderr_buffer = io.StringIO()

    with contextlib.redirect_stdout(stdout_buffer), contextlib.redirect_stderr(stderr_buffer):
        try:
            f.solve(partial_model,"1",f"{domain}.conforms")
        except Exception as e:
            pass

    message = stdout_buffer.getvalue()
    error_message = stderr_buffer.getvalue()

    return message + "\n" + error_message

solve_tool = FunctionTool.from_defaults(fn=solve)

In [6]:
def extract(task_id: str):
    """Extract the solve result from the task with task_id.
    If the partial model is solvable, will return the solution.
    Else if the partial model is unsolvable, will return the least unsatisfied core conditions.
    The task_id is a string representing an integer which is incremented by 1 each time we run the solve command."""
    
    stdout_buffer = io.StringIO()
    stderr_buffer = io.StringIO()

    with contextlib.redirect_stdout(stdout_buffer), contextlib.redirect_stderr(stderr_buffer):
        try:
            f.extract(task_id, "0", "0")
        except Exception as e:
            pass

    message = stdout_buffer.getvalue()
    error_message = stderr_buffer.getvalue()

    return message + "\n" + error_message

extract_result_tool = FunctionTool.from_defaults(fn=extract)

In [7]:
react_agent = ReActAgent.from_tools(tools=[load_tool, solve_tool, extract_result_tool], llm=llama3, verbose=True)

In [8]:
react_system_header_str = """\
You are a helpful AI agent designed to suggest repairs to unsolvable models written in a DSL called Formula.
## Work Flow
Your typical workflow is to:
1. Load a Formula script from a user-provided file path;
2. Solve the partial model in that script;
3. Extract the solve result:
    - If the model is solvable, return the solution and your task is done.
    - If the model is unsolvable:
        1. fix the model using your NLP abilities, without calling any tools
        2. return the entire code after fix
    
## Tools
You have access to four tools. You are responsible for using the tools in a correct sequence to complete the task at hand. This may require breaking the task into subtasks and using different tools to complete each subtask.

You have access to the following tools:
{tool_desc} 

## Output Format
You should choose the appropriate format based on the task at hand.

### If you need to use a tool:
```
Thought: I need to use a tool to help me answer the question.
Action: tool name (one of {tool_names})
Action Input: the input to the tool, in a JSON format representing the kwargs (e.g. {{"input": "hello world", "num_beams": 5}})
```

The response would be the tool's output in the following format:
```
Observation: tool response
```
### If you do not need to call any tool and can answer the question using you NLP ability:
```
Thought: I do not need to use any tool; Instead, I will use my NLP ability to do reasoning based on the information at hand.
Answer: [your answer here]
```

Please ensure that your responses are clear and concise, following the structure provided above.

## Current Conversation
Below is the current conversation consisting of interleaving human and assistant messages.

"""

In [9]:
react_system_prompt = PromptTemplate(react_system_header_str)
react_agent.update_prompts({"agent_worker:system_prompt": react_system_prompt})

In [10]:
prompt = """
Load, solve, and extract from file path './data/MappingExample.4ml'.
If model is solvable, return.
If model is unsolvable, fix the model and return the fixed code.
"""

In [11]:
f.load("./data/MappingExample.4ml")
f.solve("pm", "1", "Mapping.conforms")
f.extract("0", "0", "0")

> Running step 6426e558-167f-417b-a28c-4eca84b122d1. Step input: 
Load, solve, and extract from file path './data/MappingExample.4ml'.
If model is solvable, return.
If model is unsolvable, fix the model and return the fixed code.

Thought: I need to use a tool to help me answer the question.
Action: load
Action Input: {'filePath': './data/MappingExample.4ml'}
(Compiled) MappingExample.4ml
0.68s.
Observation: domain Mapping
{
  Component ::= new (id: Integer, utilization: Real).
  Processor ::= new (id: Integer).
  Mapping   ::= new (c: Component, p: Processor).

  // The utilization must be > 50
  invalidUtilization :- c is Component, c.utilization <= 50.

  badMapping :- p is Processor, 
		s = sum(0.0, { c.utilization |
			       c is Component, Mapping(c, p) }), s > 100.

  conforms no badMapping, no invalidUtilization.
}

partial model pm of Mapping
{
  c1 is Component(0, x).
  c2 is Component(1, y).
  p1 is Processor(0).
  p2 is Processor(1).
  Mapping(c1, p1).
  Mapping(c2, p1).
}

ValueError: Reached max iterations.